# 05 · Merge the mask shards
Combines the shards from notebook 04 into one dataset, so masks join photos on `path`. **Saves:** `<your username>/serpentguard-sam-masks`, private, with `masks.zip` (255 = snake), `mask_stats.csv` and `summary.json`. **Time:** a few minutes on CPU.

The account running this must be able to open every shard notebook. The team's shards ran on three accounts, so each was downloaded with its owner's account and merged. Notebook 06 uses only the masks of photos in the clean dataset.

In [ ]:
import json, zipfile
from pathlib import Path

import kagglehub
import pandas as pd

SHARD_NOTEBOOKS = [  # (notebook, its SHARD_INDEX)
    ("puneetbajajiitm/serpentguard-sam-full-shard0", 0),
    ("jayeshbansal497/serpentguard-sam-full-shard1", 1),
    ("vikas25s/serpentguard-sam-full-shard2", 2),
    ("puneetbajajiitm/serpentguard-sam-rest-shard0", 0),
    ("jayeshbansal497/serpentguard-sam-rest-shard1", 1),
    ("vikas25s/serpentguard-sam-rest-shard2", 2),
]
UPLOAD = True
OUT = Path("/kaggle/working/serpentguard-sam-masks")
OUT.mkdir(parents=True, exist_ok=True)
OWNER = kagglehub.whoami()["username"]


In [ ]:
parts, written = [], 0
with zipfile.ZipFile(OUT / "masks.zip", "w", compression=zipfile.ZIP_STORED) as dst:
    for index, (handle, shard) in enumerate(SHARD_NOTEBOOKS):
        stats = pd.read_csv(kagglehub.notebook_output_download(handle, path="sam_masks/mask_stats.csv"))
        stats["shard"] = index
        src = zipfile.ZipFile(kagglehub.notebook_output_download(handle, path=f"masks_shard{shard}.zip"))
        names = set(src.namelist())
        for path in stats.loc[stats["n_snake_boxes"] > 0, "path"]:
            stem = Path(path).stem + ".png"
            assert stem in names, (handle, stem)
            dst.writestr(str(Path(path).with_suffix(".png")), src.read(stem))
            written += 1
        parts.append(stats)
        print(handle, len(stats), "photos")

stats = pd.concat(parts)
assert stats["path"].is_unique, "shards overlap"
stats["mask_path"] = [str(Path(p).with_suffix(".png")) if n > 0 else "" for p, n in zip(stats["path"], stats["n_snake_boxes"])]
stats.to_csv(OUT / "mask_stats.csv", index=False)
has_mask = stats["n_snake_boxes"] > 0
summary = {"photos": len(stats), "with_mask": int(has_mask.sum()), "no_snake_found": int((~has_mask).sum()),
           "flagged": int(stats["flagged"].sum()), "usable_unflagged": int((has_mask & ~stats["flagged"]).sum()),
           "hand_or_glove_removed": int((stats["n_removed_boxes"] > 0).sum()), "species": int(stats["species"].nunique()),
           "shards": len(SHARD_NOTEBOOKS)}
(OUT / "summary.json").write_text(json.dumps(summary, indent=1))
summary


In [ ]:
(OUT / "README.md").write_text("""# SerpentGuard snake masks (private, course use only)

Pseudo-label snake masks for training photos (Grounding DINO and SAM 2.1, notebook 04).
masks.zip unpacks to masks/<split>/<species>/<photo>.png, the same path as the photo in the
processed dataset; 255 = snake. mask_stats.csv has one row per photo; flagged masks need a manual
check before use.
""")
if UPLOAD:
    kagglehub.dataset_upload(f"{OWNER}/serpentguard-sam-masks", str(OUT), version_notes="05_merge_sam_masks")
